# PUMA Community — Quick Demo

Run-anywhere browser demo: no installation, no tokens, no account required.


## What you'll learn

- How a PUMA submission is structured
- How to validate a submission against the v1 schema
- How to inspect metrics and sustainability data
- How to browse all community submissions
- How to submit your own results


### Step 1: Install dependencies

Only `jsonschema` is required; everything else is in the Python standard library.


In [ ]:
!pip install --quiet jsonschema==4.23.0


### Step 2: Download the v1 schema

The schema is hosted in the public repository and can be fetched anonymously.


In [ ]:
import urllib.request

SCHEMA_URL = 'https://raw.githubusercontent.com/pumacp/puma-community/main/schema/submission.v1.json'
urllib.request.urlretrieve(SCHEMA_URL, 'submission.v1.json')
print('Downloaded schema')


### Step 3: Load a sample submission

This demo uses a deterministic placeholder submission so you can see what every field looks like in practice. Field paths follow the schema: top-level run metadata lives under `run_metadata`.


In [ ]:
import json
import urllib.request

SAMPLE_URL = 'https://raw.githubusercontent.com/pumacp/puma-community/main/notebooks/sample_submission.json'
urllib.request.urlretrieve(SAMPLE_URL, 'sample_submission.json')

with open('sample_submission.json') as fh:
    submission = json.load(fh)

rm = submission['run_metadata']
print('submission_id:', submission['submission_id'])
print('scenario     :', rm['scenario'])
print('model        :', rm['model'])
print('strategy     :', rm['strategy'])
print('submitter    :', submission['submitter']['name_or_alias'])


### Step 4: Validate the submission

`jsonschema.validate()` raises `ValidationError` on the first schema-failure and returns `None` on success.


In [ ]:
import json
import jsonschema

with open('submission.v1.json') as fh:
    schema = json.load(fh)

jsonschema.validate(submission, schema)
print('Submission is valid against schema v1.0.0')


### Step 5: Inspect metrics and sustainability

Only non-null metrics are reported. The schema guarantees at least one of `f1_macro`, `mae`, or `accuracy` is present.


In [ ]:
print('--- Metrics ---')
for key, value in submission['metrics'].items():
    if value is None:
        continue
    print(f'{key:<24s} {value}')

print()
print('--- Sustainability ---')
for key, value in submission['sustainability'].items():
    print(f'{key:<24s} {value}')

print()
print('--- Integrity ---')
print('predictions_summary_hash:',
      submission['integrity']['predictions_summary_hash'][:12], '...')


### Step 6: Browse all PUMA Community submissions

The GitHub Contents API serves the `submissions/` directory anonymously. Note: the anonymous rate limit is **60 requests per hour per IP**; pass a `Authorization` header in production uses.


In [ ]:
import json
import urllib.request

req = urllib.request.Request(
    'https://api.github.com/repos/pumacp/puma-community/contents/submissions',
    headers={'Accept': 'application/vnd.github+json'},
)
with urllib.request.urlopen(req) as resp:
    contents = json.load(resp)

files = [c['name'] for c in contents if c['name'].endswith('.json')]
print(f'Found {len(files)} submissions:')
for f in files[:10]:
    print(f'  - {f}')


### Step 7: Submit your own results

The recommended path is the PUMA tool's built-in command `puma share-results`. It builds the payload from your local SQLite results, validates the schema, scans for personal data, computes the integrity hash, forks the repository, creates a branch, commits the JSON file, and opens the Pull Request — all in one command.

See [CONTRIBUTING.md](https://github.com/pumacp/puma-community/blob/main/CONTRIBUTING.md) for the full guide, including the manual submission path for advanced users.


---

**Further reading**

- Schema reference: <https://github.com/pumacp/puma-community/blob/main/schema/submission.v1.json>
- Wiki: <https://github.com/pumacp/puma-community/wiki>
- Maintainer guide: <https://github.com/pumacp/puma-community/blob/main/docs/maintainer-guide.md>
